# RAG Naive con LangChain + FAISS
Este notebook carga PDFs, los divide en fragmentos, crea embeddings y responde preguntas usando Llama.

In [ ]:
# Librerías estándar
import os

# LangChain
from langchain_ollama import OllamaLLM
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


## 1. Configuración del modelo LLM

In [ ]:
llm = OllamaLLM(model="llama3.2:1b")

c:\Users\Patricio Ricardi\PYTHON\LLM-RAG-Naive\LLM-RAG-Naive\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


## 2. Carga de documentos

In [ ]:
# Definimos la ruta de la carpeta
pdf_folder_path = "./data/"
docs = []

# Iteramos sobre los archivos en la carpeta 'data'
for file in os.listdir(pdf_folder_path):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder_path, file))
        docs.extend(loader.load())

print(f"✅ Se cargaron {len(docs)} páginas de los PDFs.")

✅ Se cargaron 6 páginas de los PDFs.


## 3. División en fragmentos

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100, # Solapamiento para no perder contexto entre trozos
    length_function=len
)

splits = text_splitter.split_documents(docs)
print(f"✅ Los PDFs se dividieron en {len(splits)} fragmentos.")

✅ Los PDFs se dividieron en 44 fragmentos.


In [4]:
# Ver el contenido del primer fragmento
if splits:
    print("Muestra del primer fragmento:")
    print(splits[0].page_content)

Muestra del primer fragmento:
REPORTE  EXCLUSIVO:  La  NASA  Confirma  que  el  "Objeto  de  las  Marianas"  no  es  Natural  
ni
 
Humano
  Un  sumergible  no  tripulado  recuperó  en  la  Fosa  de  las  Marianas  un  artefacto  de  12.000  
años
 
de
 
antigüedad
 
que
 
emite
 
pulsos
 
electromagnéticos
 
sincronizados
 
con
 
la
 
rotación
 
de
 
la
 
Tierra.


## 4. Creación de embeddings

In [ ]:
# Definimos el modelo de embeddings
print("🔄 Cargando modelo de embeddings...")
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

🔄 Cargando modelo de embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 5. Almacén vectorial FAISS

In [ ]:
# 2. Creamos el almacén vectorial (Vector Store)
# Esto toma tus 'splits' de la Fase 3 y los convierte en vectores usando el modelo
print("🧠 Generando vectores e indexando en FAISS (esto puede tardar unos segundos)...")
vectorstore = FAISS.from_documents(splits, embeddings_model)

print("✅ Almacén vectorial creado exitosamente.")

# 3. Prueba rápida de búsqueda semántica (Retriever)
query = "¿Qué objeto encontró la NASA en las Marianas?"
docs_relacionados = vectorstore.similarity_search(query, k=2)

print("\n🔍 Prueba de búsqueda:")
for i, doc in enumerate(docs_relacionados):
    print(f"Resultado {i+1}: {doc.page_content[:100]}...")

🧠 Generando vectores e indexando en FAISS (esto puede tardar unos segundos)...
✅ Almacén vectorial creado exitosamente.

🔍 Prueba de búsqueda:
Resultado 1: REPORTE  EXCLUSIVO:  La  NASA  Confirma  que  el  "Objeto  de  las  Marianas"  no  es  Natural  
ni
...
Resultado 2: de
 
Exploración
 
Oceánica
 
de
 
la
 
NOAA
 
revelaron
 
un
 
hallazgo
 
que,
 
según
 
sus
 
prop...


## 6. Definición del pipeline RAG

In [ ]:
# 1. Definimos el Prompt
template = """Eres un asistente técnico experto. Responde la pregunta basándote únicamente en el siguiente contexto:
{context}

Pregunta: {question}

Respuesta:"""

prompt = ChatPromptTemplate.from_template(template)

# 2. Configuramos el flujo (Chain)
# Este pipeline: toma la pregunta -> busca en FAISS -> llena el prompt -> le pregunta a Llama
rag_chain = (
    {"context": vectorstore.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 7. Prueba final

In [ ]:
pregunta_final = "¿Cuales son los cuatro roles identificados en la colonia de Towada?"
print(f"Preguntando: {pregunta_final}\n")

respuesta_final = rag_chain.invoke(pregunta_final)
print("--- RESPUESTA DEL RAG ---")
print(respuesta_final)

Preguntando: ¿Cuales son los cuatro roles identificados en la colonia de Towada?

--- RESPUESTA DEL RAG ---
De las informaciones proporcionadas, se puede identificar los siguientes roles:

1. Patos con mayor masa muscular en el cuello, dedicados a transportar piedras y ramas.
2. Ingenieros hidráulicos que seleccionan la ubicación de canales y supervisan el flujo de agua.
3. Señores de inspección que exhiben un comportamiento "inspecionador" repetidamente sobre las nuevas zanjas.
4. Señores de enseñanza y civilización (Dra. Voss) que identificaron cuatro roles distintos mediante marcas en las patas de los patos.
